# Module 6.7 — Security Best Practices

### Python Mastery · Track 6: Testing, Tooling & DevOps

Most security problems come from a handful of avoidable mistakes: trusting input, running dangerous functions on untrusted data, weak randomness, unsafe handling of external commands, and outdated dependencies. This module covers practical defences using the standard library, written for everyday application code rather than specialist security work.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it. The defensive techniques run fully; dependency scanning is described, as it requires network access.
- This module teaches how to protect your own programs. It does not cover attacking systems.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Validate and sanitise input before trusting it.
2. Avoid `eval`/`exec` on untrusted data and use safe alternatives.
3. Generate secure random values with the `secrets` module.
4. Run external commands safely (`shell=False`).
5. Hash and verify data with `hashlib` and `hmac`, and scan dependencies.

**Prerequisites:** Tracks 1 to 5 (especially Module 4.6), and Modules 6.1 to 6.6.

---

## Part 1 · Validate and Sanitise Input

The first rule of secure code is to never trust input from outside your program: from users, files, networks, or other systems. **Validate** it (check it meets your expectations) and reject or clean anything that does not. Validation prevents a wide range of problems, from crashes to injection attacks, by ensuring data is in the shape your code assumes.

In [ ]:
import re

def validate_username(name):
    """Accept only short alphanumeric usernames; reject everything else."""
    if not isinstance(name, str):
        raise TypeError("username must be a string")
    if not re.fullmatch(r"[A-Za-z0-9_]{3,20}", name):
        raise ValueError("username must be 3-20 letters, digits, or underscores")
    return name

for candidate in ["asha_99", "ab", "robert; DROP TABLE users", "good_name"]:
    try:
        print(f"accepted: {validate_username(candidate)!r}")
    except (ValueError, TypeError) as e:
        print(f"rejected {candidate!r}: {e}")

In [ ]:
def validate_age(raw):
    """Convert and bound-check an age coming in as text."""
    try:
        age = int(raw)
    except (ValueError, TypeError):
        raise ValueError("age must be a whole number")
    if not 0 <= age <= 150:
        raise ValueError("age out of plausible range")
    return age

for value in ["30", "-5", "abc", "200", "45"]:
    try:
        print(f"{value!r} -> {validate_age(value)}")
    except ValueError as e:
        print(f"{value!r} rejected: {e}")

## Part 2 · Avoid `eval` and `exec` on Untrusted Data

As Module 4.6 stressed, `eval` and `exec` run whatever code they are given. Passing untrusted input to them allows arbitrary, possibly destructive, code execution. For the common case of turning a string into a Python value (a number, list, or dict), use `ast.literal_eval`, which only accepts literals and cannot run code.

In [ ]:
import ast

# A configuration value arrives as text and must become a Python object.
trusted_looking_inputs = ["[1, 2, 3]", "{'limit': 10}", "(1, 2)"]
for text in trusted_looking_inputs:
    value = ast.literal_eval(text)        # safe: only literals are allowed
    print(f"{text!r} -> {value!r} ({type(value).__name__})")

# A malicious string that would run code under eval is simply rejected here.
malicious = "__import__('os').system('echo attacked')"
try:
    ast.literal_eval(malicious)
except (ValueError, SyntaxError) as e:
    print("malicious input safely rejected:", type(e).__name__)

## Part 3 · Secure Randomness with `secrets`

The `random` module is fine for simulations and games, but it is **predictable** and must never be used for anything security-sensitive such as passwords, tokens, or session identifiers. The `secrets` module generates cryptographically strong random values suitable for these uses.

In [ ]:
import secrets
import string

# A secure random token, for example a password-reset or API token.
token = secrets.token_hex(16)             # 16 random bytes as hex (32 characters)
print("secure token:", token)

# A secure URL-safe token.
url_token = secrets.token_urlsafe(16)
print("url-safe token:", url_token)

# A secure random password from a chosen alphabet.
alphabet = string.ascii_letters + string.digits
password = "".join(secrets.choice(alphabet) for _ in range(12))
print("generated password:", password)

# secrets.compare_digest compares secrets without leaking timing information.
print("constant-time compare:", secrets.compare_digest("abc123", "abc123"))

## Part 4 · Safe Subprocess Usage

Running external commands is sometimes necessary, but doing it carelessly invites **command injection**. The danger is `shell=True` combined with untrusted input: the shell interprets special characters, so a crafted argument can run extra commands. The safe approach is to pass the command as a **list of arguments** with `shell=False` (the default), so arguments are never interpreted by a shell.

In [ ]:
import subprocess

# SAFE: arguments as a list, no shell involved. The filename is treated as data,
# never as a command, even if it contains shell metacharacters.
user_supplied = "report.txt; rm -rf important"   # a hostile-looking "filename"

result = subprocess.run(
    ["echo", user_supplied],          # the whole string is a single argument
    capture_output=True, text=True,
)
print("safe call output:", result.stdout.strip())
print("the dangerous part was treated as plain text, not executed")

# The unsafe pattern, shown only to be avoided:
#   subprocess.run(f"echo {user_supplied}", shell=True)   # NEVER with untrusted input

## Part 5 · Hashing, Integrity, and Dependency Scanning

`hashlib` computes cryptographic hashes, used to verify that data has not changed (integrity) and, with the right approach, to store passwords. A plain hash is not enough for passwords; real systems use a slow, salted algorithm such as the one `hashlib.scrypt` or `pbkdf2_hmac` provides. `hmac` verifies that a message came from someone holding a shared secret and was not tampered with.

Finally, your dependencies can contain known vulnerabilities; tools such as `pip-audit` scan your installed packages against vulnerability databases.

In [ ]:
import hashlib
import hmac

# A content hash verifies integrity: the same input always gives the same digest,
# and any change produces a completely different one.
data = b"important message"
digest = hashlib.sha256(data).hexdigest()
print("sha256:", digest[:32], "...")
print("changed by one byte:", hashlib.sha256(b"important messagE").hexdigest()[:32], "...")

# Password storage uses a SALTED, SLOW hash (never a plain fast hash).
salt = b"a-unique-random-salt-per-user"
hashed_pw = hashlib.pbkdf2_hmac("sha256", b"my-password", salt, 100_000)
print("derived key length (bytes):", len(hashed_pw))

In [ ]:
import hmac
import hashlib

# HMAC verifies authenticity: only someone with the shared key can produce a valid tag.
secret_key = b"shared-secret-key"
message = b"transfer 100 to account 42"

tag = hmac.new(secret_key, message, hashlib.sha256).hexdigest()
print("message tag:", tag[:32], "...")

# The receiver recomputes the tag and compares it in constant time.
def verify(key, message, tag):
    expected = hmac.new(key, message, hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected, tag)

print("valid message verifies:", verify(secret_key, message, tag))
print("tampered message fails:", verify(secret_key, b"transfer 999 to account 99", tag))

In [ ]:
# Dependency scanning is run as a tool (requires network access to vulnerability data):
print("Scan installed dependencies for known vulnerabilities with:")
print("  pip install pip-audit")
print("  pip-audit")
print()
print("Run it in continuous integration so vulnerable dependencies fail the build.")
print("The 'safety' tool serves a similar purpose.")

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Validating an email-like string (Easy)

In [ ]:
import re

def looks_like_email(value):
    return bool(re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", value))

for candidate in ["asha@example.com", "not-an-email", "a@b.co"]:
    print(candidate, "->", looks_like_email(candidate))

### Example 2 — A secure token (Easy)

In [ ]:
import secrets
print("token:", secrets.token_hex(8))      # 8 bytes -> 16 hex characters
print("url-safe:", secrets.token_urlsafe(8))

### Example 3 — Safe parsing of a literal (Medium)

In [ ]:
import ast

def parse_setting(text):
    """Turn a config string into a Python value, safely."""
    try:
        return ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return None                       # reject anything that is not a plain literal

print(parse_setting("[1, 2, 3]"))
print(parse_setting("{'on': True}"))
print(parse_setting("os.system('bad')"))  # not a literal -> None

### Example 4 — A safe external command (Medium)

In [ ]:
import subprocess

def word_count(text):
    # Pass the text as an argument, not through a shell.
    result = subprocess.run(["wc", "-w"], input=text, capture_output=True, text=True)
    return int(result.stdout.strip())

print("word count:", word_count("one two three four"))

### Example 5 — Verifying a download with a hash (Difficult)

In [ ]:
import hashlib

def file_hash(content: bytes):
    return hashlib.sha256(content).hexdigest()

# Imagine a publisher provides this expected hash alongside a download.
original = b"the genuine file contents"
expected = file_hash(original)

# A consumer checks what they received against the expected hash.
received_good = b"the genuine file contents"
received_bad = b"the genuine file contents (tampered)"

print("genuine matches:", file_hash(received_good) == expected)
print("tampered matches:", file_hash(received_bad) == expected)

### Example 6 — A signed message with HMAC (Difficult)

In [ ]:
import hmac, hashlib

class SignedMessenger:
    """Sign and verify messages with a shared secret key."""
    def __init__(self, key: bytes):
        self.key = key

    def sign(self, message: bytes):
        tag = hmac.new(self.key, message, hashlib.sha256).hexdigest()
        return message, tag

    def verify(self, message: bytes, tag: str):
        expected = hmac.new(self.key, message, hashlib.sha256).hexdigest()
        return hmac.compare_digest(expected, tag)

m = SignedMessenger(b"top-secret-key")
message, tag = m.sign(b"approve payment")

print("authentic message verifies:", m.verify(message, tag))
print("forged message rejected:", m.verify(b"approve different payment", tag))

---

## Exercises

**Exercise 1 (Easy).** Write a function `validate_pin(pin)` that accepts only a string of exactly four digits, returning `True` or `False`. Test it on a few inputs.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Use `secrets` to generate a 12-character URL-safe token and a random 6-digit numeric code (use `secrets.randbelow`).

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Write a function that safely turns a string into a Python list using `ast.literal_eval`, returning an empty list if the input is not a valid literal. Test it with a valid list string and a malicious string.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Use `subprocess.run` with a list of arguments (not a shell string) to call `echo` with a user-supplied string that contains shell metacharacters, and show the characters are treated as plain text.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Write a function that hashes a password with a salt using `hashlib.pbkdf2_hmac`, and a second function that verifies a candidate password against the stored salt and hash (using `hmac.compare_digest`). Demonstrate a correct and an incorrect password.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import re
def validate_pin(pin):
    return bool(re.fullmatch(r"\d{4}", pin))

for p in ["1234", "12", "abcd", "99999"]:
    print(p, "->", validate_pin(p))

**Solution 2**

In [ ]:
import secrets
print("token:", secrets.token_urlsafe(9)[:12])
code = "".join(str(secrets.randbelow(10)) for _ in range(6))
print("6-digit code:", code)

**Solution 3**

In [ ]:
import ast
def parse_list(text):
    try:
        value = ast.literal_eval(text)
        return value if isinstance(value, list) else []
    except (ValueError, SyntaxError):
        return []

print(parse_list("[1, 2, 3]"))
print(parse_list("__import__('os')"))

**Solution 4**

In [ ]:
import subprocess
hostile = "hello && echo INJECTED"
result = subprocess.run(["echo", hostile], capture_output=True, text=True)
print(result.stdout.strip())
print("the '&&' was printed, not executed")

**Solution 5**

In [ ]:
import hashlib, hmac, secrets

def hash_password(password, salt=None):
    salt = salt or secrets.token_bytes(16)
    digest = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, 100_000)
    return salt, digest

def verify_password(candidate, salt, digest):
    test = hashlib.pbkdf2_hmac("sha256", candidate.encode(), salt, 100_000)
    return hmac.compare_digest(test, digest)

salt, stored = hash_password("correct horse")
print("correct password:", verify_password("correct horse", salt, stored))
print("wrong password:", verify_password("wrong guess", salt, stored))

---

## Summary and Key Points

- Never trust external input: validate it against strict expectations and reject or sanitise anything that does not fit, which prevents crashes and injection attacks.
- Do not run `eval` or `exec` on untrusted data; use `ast.literal_eval` to turn strings into literal Python values safely.
- Use `secrets`, not `random`, for security-sensitive values such as tokens and passwords, and compare secrets with `secrets.compare_digest` to avoid timing leaks.
- Run external commands with a list of arguments and `shell=False` (the default) to prevent command injection; never combine `shell=True` with untrusted input.
- Use `hashlib` for integrity (and a salted, slow algorithm like `pbkdf2_hmac` for passwords), `hmac` for authenticity with a shared key, and scan dependencies with `pip-audit` in continuous integration.

### Next module: 6.8 — Code Review & Pythonic Excellence

The next module covers writing readable, maintainable code, conducting effective code reviews, and practical refactoring techniques.